# 01_baseline

## 1. Setup and configuration

In [ ]:
# --- Global configuration & seed -----------------------------------------------
import os, sys, json, random, platform, hashlib
from datetime import datetime
from pathlib import Path

SEED = 42
N_SAMPLES = 500
N_ONESHOT_ITERS = 3
PROMPTS = ["domain"]

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

COMPUTE_ROC_AUC = True

# --- Automatic output directory ------------------------------------------------
# RUN_NAME = f"HUMAID_{N_SAMPLES}_run"
RUN_NAME = "HUMAID_500_run"
RUN_DIR = Path.cwd() / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

def run_path(filename):
    """Return an absolute path inside this run's output directory."""
    return str(RUN_DIR / filename)

print(f"RUN_DIR={RUN_DIR.resolve()}")

print(
    f"SEED={SEED} | N_SAMPLES={N_SAMPLES} | "
    f"one-shot iters={N_ONESHOT_ITERS} | prompts={PROMPTS}"
)

In [ ]:
# --- Determinism ------------------------------------------------------------------
def set_all_seeds(seed: int):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np; np.random.seed(seed)
    except Exception: pass
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception: pass

set_all_seeds(SEED)
print("Seeds set for random / numpy / torch.")


In [ ]:
# --- Reproducibility record -------------------------------------------------------
def _pkg_version(name):
    try:
        from importlib.metadata import version; return version(name)
    except Exception:
        return "not installed"

record = {"timestamp_utc": datetime.utcnow().isoformat() + "Z",
          "python_version": sys.version.split()[0],
          "platform": platform.platform(),
          "processor": platform.processor() or "unknown", "seed": SEED}
for pkg in ["torch","transformers","accelerate","datasets","numpy","pandas","scikit-learn","tqdm"]:
    record[f"pkg::{pkg}"] = _pkg_version(pkg)
try:
    import torch
    record["cuda_available"] = torch.cuda.is_available()
    record["cuda_version"] = getattr(torch.version, "cuda", None)
    record["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU only)"
except Exception as e:
    record["cuda_available"] = False; record["gpu"] = f"torch unavailable: {e}"

import pandas as pd
pd.DataFrame(sorted(record.items()), columns=["key", "value"])


## 2. Model: `Qwen2.5-3B-Instruct` 

In [ ]:
# --- Load Qwen2.5-3B-Instruct -----------------------------------------------------
# pip install "transformers>=4.43" accelerate torch datasets scikit-learn tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=_dtype,
    device_map="auto" if torch.cuda.is_available() else None)
model.eval()
DEVICE = next(model.parameters()).device
print(f"Loaded {MODEL_NAME} on {DEVICE} (dtype={_dtype}).")


In [ ]:
# --- Generation config ------------------------------------------------------------
# [UNSPEC] paper says only "default model settings". Greedy => deterministic/reproducible.
GEN_KWARGS = dict(do_sample=False, num_beams=1, max_new_tokens=12,
                  pad_token_id=tokenizer.eos_token_id)
print("Generation kwargs:", GEN_KWARGS)


## 3. Dataset — HUMAID only

For the current research pipeline:
- **TEST** is used for one-shot evaluation.
- **TRAIN** is used only to draw one-shot exemplars.
- `N_SAMPLES = 500` by default for a quick experimental run.


In [ ]:
# --- HUMAID source ---------------------------------------------------------------
HF = {
    "HUMAID_repo": "QCRI/HUMAID-all",
}

print(json.dumps(HF, indent=2))


## 4. Dataset loading


In [ ]:
# --- Shared helpers ---------------------------------------------------------------
import pandas as pd
from datasets import load_dataset, concatenate_datasets

def _concat_all(ds):
    return concatenate_datasets([ds[s] for s in ds.keys()])

def _find_col(cols, cands):
    low = {c.lower().strip(): c for c in cols}
    for c in cands:
        if c in low: return low[c]
    for c in cands:
        for lc, orig in low.items():
            if c in lc: return orig
    return None

TEXT_CANDS  = ["tweet_text", "text", "tweet", "message", "content"]
LABEL_CANDS = ["class_label", "label", "category", "class"]
ID_CANDS    = ["tweet_id", "id", "tweetid"]


In [ ]:
# --- HUMAID loader (QCRI/HUMAID-all) — SPLIT-PRESERVING -------------------------
# [FIX] The paper (Sec. 3.3) draws one-shot exemplars from the TRAIN split and evaluates on the
# TEST split. The previous version merged every split with _concat_all(), destroying that boundary.
# We now return the official train/test splits separately.
def _HUMAID_split_to_frame(split_ds):
    raw = split_ds.to_pandas().astype(str)
    tcol = _find_col(raw.columns, TEXT_CANDS)
    lcol = _find_col(raw.columns, LABEL_CANDS)
    icol = _find_col(raw.columns, ID_CANDS)
    out = pd.DataFrame({"id": raw[icol] if icol else raw.index.astype(str),
                        "text": raw[tcol], "label": raw[lcol]})
    out.attrs["encoding"] = "utf-8 (HF QCRI/HUMAID-all)"
    return out

def load_HUMAID():
    ds = load_dataset(HF["HUMAID_repo"])
    print(f"[HUMAID] splits: {list(ds.keys())}")
    train = _HUMAID_split_to_frame(ds["train"])
    test  = _HUMAID_split_to_frame(ds["test"])
    print(f"[HUMAID] official split -> train {len(train)} | test {len(test)}")
    return {"train": train, "test": test}


In [ ]:
def load_HUMAID():
    ds = load_dataset(
        HF["HUMAID_repo"],
        verification_mode="no_checks"
    )

    print(f"[HUMAID] splits: {list(ds.keys())}")

    train = _HUMAID_split_to_frame(ds["train"])
    validation = _HUMAID_split_to_frame(ds["validation"])
    test = _HUMAID_split_to_frame(ds["test"])

    return {
        "train": train,
        "validation": validation,
        "test": test
    }

In [ ]:
# --- Load HUMAID only ------------------------------------------------------------
LOADERS = {"HUMAID": load_HUMAID}

raw_loaded = {}
for name, fn in LOADERS.items():
    raw_loaded[name] = fn()
    print(f"[OK] {name}: { {k: len(v) for k, v in raw_loaded[name].items()} }")


### 5. Build split-aware HUMAID structures

In [ ]:
# --- Assemble HUMAID split-aware structures -------------------------------------
raw_train = {
    "HUMAID": raw_loaded["HUMAID"]["train"].reset_index(drop=True)
}

raw_test = {
    "HUMAID": raw_loaded["HUMAID"]["test"].reset_index(drop=True)
}

# Union is used only for label-space discovery and diagnostics.
raw_datasets = {
    "HUMAID": pd.concat(
        [raw_train["HUMAID"], raw_test["HUMAID"]],
        ignore_index=True
    )
}

print(
    f"[HUMAID] TRAIN={len(raw_train['HUMAID'])} | "
    f"TEST={len(raw_test['HUMAID'])} | "
    f"label space={raw_datasets['HUMAID']['label'].nunique()} classes"
)


In [ ]:
# --- Verification report (ALL numbers computed here) ------------------------------
def verify_report(name, df):
    text = df["text"].astype(str)
    lw = text.str.split().map(len); lc = text.str.len()
    return {"dataset": name,
            "encoding": df.attrs.get("encoding", "unknown"),
            "total_samples": len(df),
            "n_classes": df["label"].nunique(),
            "missing_label_rows": int((df["label"].astype(str).str.strip() == "").sum()),
            "empty_text_rows": int((text.str.strip() == "").sum()),
            "duplicate_text_rows": int(df.duplicated(subset=["text"]).sum()),
            "rows_with_non_ascii": int(text.map(lambda s: any(ord(c) > 127 for c in s)).sum()),
            "len_words_min": int(lw.min()), "len_words_mean": round(float(lw.mean()), 2),
            "len_words_max": int(lw.max()),
            "len_chars_min": int(lc.min()), "len_chars_mean": round(float(lc.mean()), 2),
            "len_chars_max": int(lc.max())}

pd.DataFrame([verify_report(n, d) for n, d in raw_datasets.items()]).set_index("dataset").T


In [ ]:
# --- Per-dataset label inventory & distribution (computed) ------------------------
for name, df in raw_datasets.items():
    print(f"\n=== {name}: label distribution (from data) ===")
    vc = df["label"].value_counts(dropna=False)
    display(pd.DataFrame({"count": vc, "proportion": (vc / vc.sum()).round(4)}))


In [ ]:
import numpy as np

def clean_pool(df):
    d = df.copy()
    d["text"] = d["text"].astype(str); d["label"] = d["label"].astype(str)
    d = d[(d["text"].str.strip() != "") & (d["label"].str.strip() != "")]
    d = d.drop_duplicates(subset=["text"])
    return d.sort_values(["id", "text"], kind="mergesort").reset_index(drop=True)

# [FIX] Evaluation is drawn ONLY from the TEST split; one-shot exemplars (Section 8) come ONLY
# from the TRAIN split — exactly as the paper specifies (Sec. 3.3).
# N_SAMPLES=None -> evaluate on the entire clean TEST split (fully paper-faithful, slower).
def sample_eval(df, seed=SEED, n=N_SAMPLES):
    pool = clean_pool(df)
    if n is None or len(pool) <= n:
        return pool.reset_index(drop=True), pool
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(pool), size=n, replace=False))
    return pool.iloc[idx].reset_index(drop=True), pool

samples, test_pools, train_pools = {}, {}, {}
for name in raw_test:
    samples[name], test_pools[name] = sample_eval(raw_test[name], seed=SEED, n=N_SAMPLES)  # eval <- TEST
    train_pools[name] = clean_pool(raw_train[name])                                         # exemplars <- TRAIN
    print(f"[{name}] eval={len(samples[name])} (from TEST {len(test_pools[name])}) | "
          f"TRAIN exemplar pool={len(train_pools[name])}")


In [ ]:
# --- Reproducibility check + auditable fingerprint --------------------------------
for name in raw_test:
    a, _ = sample_eval(raw_test[name], seed=SEED); b, _ = sample_eval(raw_test[name], seed=SEED)
    assert a.equals(b), f"Non-reproducible sampling for {name}!"
    fp = hashlib.sha256("|".join(a["id"].astype(str)).encode()).hexdigest()[:16]
    print(f"[{name}] reproducible OK | eval-id sha256[:16] = {fp}")


In [ ]:
# --- Class coverage inside each 500-sample subset ---------------------------------
for name, s in samples.items():
    print(f"\n=== {name}: classes present in the 500-sample subset ===")
    display(s["label"].value_counts())


## 6. Prompt construction — domain

In [ ]:

# --- Label space and Domain Prompt ---------------------------------------------

def verbalize(label):
    return label.replace("_", " ").replace("-", " ").strip().lower()

def build_label_space(df):
    labels = sorted(df["label"].astype(str).unique())
    verb = {l: verbalize(l) for l in labels}
    rev = {v: k for k, v in verb.items()}
    return labels, verb, rev

def options_block(verb):
    return "; ".join(sorted(verb.values()))

def make_instruction(prompt_type, verb):
    opts = options_block(verb)

    if prompt_type == "domain":
        return (
            "The following text is a tweet posted during a disaster or crisis event. "
            f"Classify it into exactly one of the following humanitarian categories: {opts}. "
            "Return only the category name and nothing else."
        )

    raise ValueError(prompt_type)


# No system message.
# The task instruction is provided through the Domain Prompt.
# Non-matching outputs are handled by parse_output().
# The Qwen chat template is applied in generate_label().


In [ ]:
def one_shot_messages(prompt_type, verb, text, exemplars):
    # [PAPER] one-shot: the SAME instruction, then one labelled example per class, then the query.
    # Instruction first, in-context examples in the middle, query last.
    instr = make_instruction(prompt_type, verb)
    shots = "\n\n".join(f"Text: {t}\nCategory: {verb[l]}" for t, l in exemplars)
    return [{"role": "user",
             "content": f"{instr}\n\nExamples:\n{shots}\n\nText: {text}\n\nCategory:"}]


## 7. In-Context Learning inference


In [ ]:
import re, torch
from difflib import get_close_matches
def parse_output(gen_text, rev):
    g = gen_text.strip().lower()
    g = re.sub(r"^(category|answer|label)\s*[:\-]\s*", "", g).strip(" .\"'`\n")

    # 1) Existing behavior: verbalized label
    # e.g. "caution and advice"
    if g in rev:
        return rev[g]

    # 2) NEW: accept canonical dataset label directly
    # e.g. "caution_and_advice"
    canonical_lookup = {
        str(raw).strip().lower(): raw
        for raw in rev.values()
    }

    if g in canonical_lookup:
        return canonical_lookup[g]

    # 3) Existing substring matching
    for v, raw in rev.items():
        if v and v in g:
            return raw

    # 4) NEW: canonical label occurring inside a longer output
    for canonical, raw in canonical_lookup.items():
        if canonical and canonical in g:
            return raw

    # 5) Existing fuzzy matching on verbalized labels
    close = get_close_matches(
        g,
        list(rev.keys()),
        n=1,
        cutoff=0.90
    )

    if close:
        return rev[close[0]]

    # 6) NEW: fuzzy matching on canonical labels
    close_canonical = get_close_matches(
        g,
        list(canonical_lookup.keys()),
        n=1,
        cutoff=0.90
    )

    return canonical_lookup[close_canonical[0]] if close_canonical else None

@torch.no_grad()
def generate_label(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(**enc, **GEN_KWARGS)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)


In [ ]:
@torch.no_grad()
def label_logprob(messages, label_text):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    p_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    a_ids = tokenizer(" " + label_text, return_tensors="pt",
                      add_special_tokens=False).input_ids.to(DEVICE)
    full = torch.cat([p_ids, a_ids], dim=1)
    logits = model(full).logits
    tgt = full[:, 1:]
    lp = torch.log_softmax(logits[:, :-1, :].float(), dim=-1)
    tok_lp = lp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return float(tok_lp[:, p_ids.shape[1]-1:].mean().item())

def class_probabilities(messages_fn, labels, verb):
    s = np.array([label_logprob(messages_fn(), verb[l]) for l in labels])
    s = s - s.max(); p = np.exp(s); return p / p.sum()


In [ ]:

# --- One-shot Domain Inference and Per-instance Predictions ---------------------

from tqdm.auto import tqdm

# Every individual prediction is stored here for later FiNE / Ripple / Protect analysis.
PER_INSTANCE_RECORDS = []


def exemplars_for(name, iteration, labels):
    # One exemplar per label, sampled only from TRAIN.
    train_pool = train_pools[name]
    eval_texts = set(samples[name]["text"].astype(str))
    avail = train_pool[~train_pool["text"].astype(str).isin(eval_texts)]

    rng = np.random.default_rng(SEED + 1000 * iteration)
    chosen = []

    for lab in labels:
        cand = avail[avail["label"] == lab]

        if len(cand) == 0:
            continue

        row = cand.iloc[int(rng.integers(0, len(cand)))]
        chosen.append((row["text"], lab))

    return chosen


def run_setting(name, shot, prompt_type, iteration=0):
    df = samples[name]
    labels, verb, rev = build_label_space(raw_datasets[name])
    exemplars = exemplars_for(name, iteration, labels) if shot == "one" else None

    y_true, y_pred, wrong = [], [], 0
    proba = [] if COMPUTE_ROC_AUC else None

    for _, r in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"{name}|{shot}|{prompt_type}|it{iteration}",
        leave=False
    ):
        messages = one_shot_messages(
            prompt_type, verb, r["text"], exemplars
        )

        raw_output = generate_label(messages)
        pred = parse_output(raw_output, rev)

        invalid_output = pred is None

        if invalid_output:
            wrong += 1
            pred = "__wrong__"

        true_label = str(r["label"])
        y_true.append(true_label)
        y_pred.append(pred)

        if COMPUTE_ROC_AUC:
            # Reuse the exact same message construction for label-likelihood probabilities.
            msg_fn = lambda m=messages: m
            proba.append(class_probabilities(msg_fn, labels, verb))

        PER_INSTANCE_RECORDS.append({
            "dataset": name,
            "sample_id": str(r["id"]),
            "text": str(r["text"]),
            "true_label": true_label,
            "predicted_label": pred,
            "is_correct": bool(pred == true_label),
            "is_misclassified": bool(pred != true_label),
            "invalid_model_output": bool(invalid_output),
            "shot": shot,
            "prompt": prompt_type,
            "iteration": int(iteration),
            "raw_model_output": str(raw_output),
            "exact_messages_json": json.dumps(
                messages, ensure_ascii=False
            ),
            "exemplars_json": (
                json.dumps(exemplars, ensure_ascii=False)
                if exemplars is not None else "[]"
            ),
        })

    return {
        "labels": labels,
        "y_true": y_true,
        "y_pred": y_pred,
        "wrong_label_fraction": wrong / len(df),
        "proba": (np.vstack(proba) if COMPUTE_ROC_AUC else None),
    }


## 8. Metrics

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)

def compute_metrics(res):
    labels, yt, yp = res["labels"], res["y_true"], res["y_pred"]
    m = {"accuracy":  accuracy_score(yt, yp),
         "precision": precision_score(yt, yp, labels=labels, average="macro", zero_division=0),
         "recall":    recall_score(yt, yp, labels=labels, average="macro", zero_division=0),
         "f1":        f1_score(yt, yp, labels=labels, average="macro", zero_division=0),
         "f1_micro":  f1_score(yt, yp, labels=labels, average="micro", zero_division=0),
         "mcc":       matthews_corrcoef(yt, yp),
         "wrong_label_fraction": res["wrong_label_fraction"]}
    if res["proba"] is not None:
        present = [i for i, l in enumerate(labels) if l in set(yt)]
        try:
            yidx = np.array([labels.index(t) for t in yt])
            m["roc_auc"] = roc_auc_score(yidx, res["proba"][:, present],
                                         labels=present, multi_class="ovr", average="macro")
        except Exception as e:
            m["roc_auc"] = float("nan"); print(f"[ROC-AUC n/a] {e}")
    else:
        m["roc_auc"] = float("nan")
    return m

REPORT_COLS  = ["accuracy", "precision", "recall", "f1", "mcc", "roc_auc"]
NUMERIC_COLS = REPORT_COLS + ["f1_micro", "wrong_label_fraction"]


## 9. Run HUMAID experiments

In [ ]:
def evaluate_dataset(name):
    rows = []

    # One-shot
    for pt in PROMPTS:
        for it in range(N_ONESHOT_ITERS):
            r = compute_metrics(run_setting(name, "one", pt, it))
            r.update(dataset=name, shot="one", prompt=pt, iter=it)
            rows.append(r)

    return pd.DataFrame(rows)


# Prevent duplicate per-instance rows when rerunning this cell.
PER_INSTANCE_RECORDS.clear()

print("\n########## Evaluating HUMAID ##########")
# [SCOPE] Do NOT call evaluate_dataset(); execute ONLY one-shot / domain / iteration 0.
r = compute_metrics(run_setting("HUMAID", "one", "domain", 0))
r.update(dataset="HUMAID", shot="one", prompt="domain", iter=0)
all_results = {
    "HUMAID": pd.DataFrame([r])
}

results_df = pd.concat(all_results.values(), ignore_index=True)

print(f"\nCollected {len(PER_INSTANCE_RECORDS)} per-instance prediction records.")
results_df


## 10. Per-dataset results tables (one-shot)

In [ ]:

# ============================================================
# ONE-SHOT RESULTS TABLE — DOMAIN PROMPT (ITERATION 0)
# ============================================================

def dataset_tables(results_df, dataset):
    sub = results_df[results_df["dataset"] == dataset]

    # one-shot: mean over iterations per prompt + AVG across prompts
    o = sub[sub["shot"] == "one"].groupby("prompt")[REPORT_COLS].mean()
    o = pd.concat([o, o.mean().to_frame().T.rename(index={0: "AVG"})])

    return o.round(4)
# Display One-shot results for each dataset
for name in all_results:
    ot = dataset_tables(results_df, name)
    print(f"\n================= {name} — ONE-SHOT =================")
    display(ot)


In [ ]:
import pandas as pd, numpy as np

METRIC_MAP   = {"accuracy":"Accuracy","precision":"Precision","recall":"Recall",
                "f1":"Macro-F1","mcc":"MCC","roc_auc":"ROC-AUC"}
METRIC_ORDER = ["Accuracy","Precision","Recall","Macro-F1","MCC","ROC-AUC"]
PROMPT_MAP   = {"domain":"domain"}
PROMPT_ORDER = ["domain"]
SHOT_MAP     = {"one":"One"}
DATASET_ORDER = ["HUMAID"]
SHOT_ORDER    = ["One"]
metric_cols   = list(METRIC_MAP.keys())

# 1) One-shot + Domain only
agg = (results_df[
           (results_df["shot"] == "one") &
           (results_df["prompt"] == "domain")
       ]
       .groupby(["dataset","shot","prompt"], as_index=False)[metric_cols]
       .mean())

# 2) long form
long = agg.melt(id_vars=["dataset","shot","prompt"], value_vars=metric_cols,
                var_name="metric", value_name="value")
long["Metric"] = long["metric"].map(METRIC_MAP)
long["Prompt"] = long["prompt"].map(PROMPT_MAP).fillna(long["prompt"])
long["col"]    = long["dataset"] + " " + long["shot"].map(SHOT_MAP)

# 3) rows = (Metric, Prompt); columns = "dataset shot"
wide = long.pivot_table(index=["Metric","Prompt"], columns="col", values="value")

# 4) column order
col_order = [f"{d} {s}" for d in DATASET_ORDER for s in SHOT_ORDER]
wide = wide.reindex(columns=[c for c in col_order if c in wide.columns])

# 5) row order: domain only
wide = wide.reindex(pd.MultiIndex.from_product([METRIC_ORDER, PROMPT_ORDER],
                                               names=["Metric","Prompt"])).round(4)

# 6) Prompt & Metric as rightmost columns
table = wide.reset_index()
data_cols = [c for c in table.columns if c not in ("Metric","Prompt")]
table = table[data_cols + ["Prompt","Metric"]]
table["Metric"] = table["Metric"].mask(table["Metric"].duplicated(), "")

try:
    display(table.style.hide(axis="index"))
except Exception:
    display(table)

table.to_csv(run_path("comparison_table_oneshot_domain.csv"), index=False)

In [ ]:
# --- Persist HUMAID One-shot Domain aggregate + per-instance outputs ------------

if not results_df.empty:

    # One-shot + Domain only
    results_df = results_df[
        (results_df["shot"] == "one") &
        (results_df["prompt"] == "domain")
    ].copy()

    # Aggregate results
    results_df.to_csv(
        run_path("HUMAID_results_raw_per_prompt_per_iter.csv"),
        index=False
    )

    ot = dataset_tables(results_df, "HUMAID")
    ot.to_csv(run_path("HUMAID_one_shot_metrics.csv"))

    # Per-instance predictions
    pred_df = pd.DataFrame(PER_INSTANCE_RECORDS)

    pred_df = pred_df[
        (pred_df["shot"] == "one") &
        (pred_df["prompt"] == "domain")
    ].copy()

    pred_df.to_csv(
        run_path("HUMAID_all_predictions_per_instance.csv"),
        index=False
    )

    # All misclassifications
    mis_df = pred_df[pred_df["is_misclassified"]].copy()
    mis_df.to_csv(
        run_path("HUMAID_all_misclassifications.csv"),
        index=False
    )

    # Recommended downstream subset:
    # One-shot + Domain, preserving all available iterations.
    one_domain = pred_df.copy()

    one_domain.to_csv(
        run_path("HUMAID_one_shot_domain_predictions.csv"),
        index=False
    )

    one_domain_mis = one_domain[
        one_domain["is_misclassified"]
    ].copy()

    one_domain_mis.to_csv(
        run_path("HUMAID_one_shot_domain_misclassifications.csv"),
        index=False
    )

    print("Saved:")
    print("  HUMAID_results_raw_per_prompt_per_iter.csv")
    print("  HUMAID_one_shot_metrics.csv")
    print("  HUMAID_all_predictions_per_instance.csv")
    print("  HUMAID_all_misclassifications.csv")
    print("  HUMAID_one_shot_domain_predictions.csv")
    print("  HUMAID_one_shot_domain_misclassifications.csv")

else:
    print("Nothing to save yet — run Section 10 first.")